Imports

In [ ]:
import matplotlib.pyplot as plt

from src.data import load_movielens
from src.content_based import ContentBasedRecommender
from src.popularity import PopularityRecommender
from src.evaluation import train_test_split_by_time, compare_models

Load Dataset

In [ ]:
ratings, movies, users = load_movielens()

Initialize the Model

In [ ]:
model = ContentBasedRecommender(movies)

Generate Recommendations

In [ ]:
recommendations = model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(recommendations)

Inspect Results

In [ ]:
print(f"Recommendations generated: {len(recommendations)}")
display(recommendations)

## Popularity-Based Recommendations

The baseline recommends movies with at least `min_ratings` ratings and the highest share of positive (>= threshold) ratings. Every user gets the same list, minus movies they've already rated.

In [ ]:
popularity_model = PopularityRecommender(movies, min_ratings=50)

popular_recommendations = popularity_model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(popular_recommendations)

## Evaluation: Content-Based vs. Popularity

Each user's most recent 20% of ratings are held out as a test set. Both models only see the training ratings, and we check how many of their top-10 recommendations are movies the user went on to rate 4 or higher.

- **precision@10**: share of the 10 recommendations the user liked
- **recall@10**: share of the user's liked held-out movies that were recommended
- **hit_rate@10**: share of users with at least one liked movie in their top 10
- **coverage**: share of the whole catalog recommended to at least one user

In [ ]:
train, test = train_test_split_by_time(ratings, test_fraction=0.2)

print(f"Train ratings: {len(train):,}")
print(f"Test ratings:  {len(test):,}")

In [ ]:
# Popularity is ranked from the training ratings only, so test ratings never leak in.
models = {
    "Content-based (genres)": ContentBasedRecommender(movies),
    "Popularity": PopularityRecommender(movies, min_ratings=50),
}

comparison = compare_models(models, train, test, k=10, threshold=4)
display(comparison.round(4))

In [ ]:
metrics = ["precision@10", "recall@10", "hit_rate@10", "coverage"]

fig, axes = plt.subplots(1, len(metrics), figsize=(14, 4))
for ax, metric in zip(axes, metrics):
    comparison[metric].plot.bar(ax=ax, color=["#4C72B0", "#DD8452"])
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("Content-based vs. popularity recommendations")
plt.tight_layout()
plt.show()

### Results

[Which approach scored higher on precision/recall/hit rate, and which covered more of the catalog? What does that suggest about balancing popularity and personal preference?]